In [1]:
import nltk
import textstat
import polars as pl
import numpy as np

from collections import Counter
from datasets import load_dataset
from nltk.tokenize import word_tokenize

/home/ubuntu/miniconda/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
nltk.download("punkt")
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/ubuntu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ubuntu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
load_dataset("roneneldan/TinyStories", split="train").to_parquet("tinystories.parquet")
load_dataset("SimpleStories/SimpleStories", split="train").to_parquet("simplestories.parquet")

Creating parquet from Arrow format: 100%|██████████| 32/32 [00:28<00:00,  1.11ba/s]


3142783327

In [4]:
tinystories_df = pl.scan_parquet("tinystories.parquet")
simplestories_df = pl.scan_parquet("simplestories.parquet")

In [15]:
tinystories_df_sample = tinystories_df.collect().sample(1000, seed=1)
simplestories_df_sample = simplestories_df.collect().sample(1000, seed=1)

In [16]:
def calculate_flesch_kincaid(df, text_column):
    """
    Calculate Flesch-Kincaid grade level for texts in a Polars DataFrame.
    
    Args:
        df: Polars DataFrame
        text_column: Name of the column containing text
    
    Returns:
        DataFrame with added 'flesch_kincaid_grade' column
    """
    return df.with_columns(
        pl.col(text_column)
        .map_elements(lambda text: textstat.flesch_kincaid_grade(text), return_dtype=pl.Float64)
        .alias('flesch_kincaid_grade')
    )

In [17]:
tinystories_df_sample = tinystories_df_sample.with_columns(
    pl.col("text").str.split(" ").list.len().alias("word_count")
)

In [18]:
tinystories_df_sample = calculate_flesch_kincaid(tinystories_df_sample, 'text')
simplestories_df_sample = calculate_flesch_kincaid(simplestories_df_sample, 'story')

In [19]:
ts_result = tinystories_df_sample.select(
    word_count_mean = pl.col("word_count").mean(),
    word_count_std = pl.col("word_count").std(),
    flesch_kincaid_mean = pl.col("flesch_kincaid_grade").mean(),
    flesch_kincaid_std = pl.col("flesch_kincaid_grade").std(),
    
)

ss_result = simplestories_df_sample.select(
    word_count_mean = pl.col("word_count").mean(),
    word_count_std = pl.col("word_count").std(),
    flesch_kincaid_mean = pl.col("flesch_kincaid_grade").mean(),
    flesch_kincaid_std = pl.col("flesch_kincaid_grade").std(),
    
)

In [20]:
ts_result

word_count_mean,word_count_std,flesch_kincaid_mean,flesch_kincaid_std
f64,f64,f64,f64
170.887,76.142432,3.211245,1.445278


In [21]:
ss_result

word_count_mean,word_count_std,flesch_kincaid_mean,flesch_kincaid_std
f64,f64,f64,f64
275.898,132.529037,3.772963,1.243547


## Compression ratio and Self-BLEU homogenization score

In [14]:
# random subsample from each 
from diversity import (
    compression_ratio,
    homogenization_score,
    ngram_diversity_score,
)

[nltk_data] Downloading package punkt_tab to /home/ubuntu/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [22]:
ts_cr = compression_ratio(tinystories_df_sample["text"], algorithm='gzip')
print(f"Tiny stories Compression Ratio: {ts_cr:.4f}")

ss_cr = compression_ratio(simplestories_df_sample["story"], algorithm='gzip')
print(f"Simple stories Compression Ratio: {ss_cr:.4f}")

Tiny stories Compression Ratio: 3.2000
Simple stories Compression Ratio: 2.9590


### homogenization_score

Takes ~50 mins to run all the way through

In [23]:
ts_hs = homogenization_score(tinystories_df_sample["text"], measure='bleu')
print(f"Tiny stories Homogenization score: {ts_hs:.4f}")

ss_hs = homogenization_score(simplestories_df_sample["story"], measure='bleu')
print(f"Simple stories Homogenization score: {ss_cr:.4f}")

==> Scoring all pairs


100%|██████████| 1000/1000 [18:33<00:00,  1.11s/it]


Tiny stories Compression Ratio: 3.2000
==> Scoring all pairs


100%|██████████| 1000/1000 [25:04<00:00,  1.50s/it]

Simple stories Compression Ratio: 2.9590


In [ ]:
print(f"Tiny stories Homogenization score: {ts_hs:.4f}")
print(f"Simple stories Homogenization score: {ss_hs:.4f}")

## N-gram diversity score

In [ ]:
ts_n_gram_results = {}
ss_n_gram_results = {}

for n in range(1, 11, 1):
    ngd = ngram_diversity_score(tinystories_df_sample["text"], n=n)
    ts_n_gram_results[n] = ngd
    ngd = ngram_diversity_score(simplestories_df_sample["story"], n=n)
    ss_n_gram_results[n] = ngd

print("Tiny stories n-gram")
print(ts_n_gram_results)
print("Simple stories n-gram")
print(ss_n_gram_results)